In [1]:
from collections import defaultdict
import random
from random import Random
import os
import pickle as pkl
from pprint import pprint
import time

import numpy as np
import pandas as pd
# from sklearn.model_selection import train_test_split
from tqdm import tqdm

from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold

# from rdmc import RDKitMol

import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
path = '../data/ccsdtf12/canonicalized_smiles/ccsdtf12_dz_canonical.csv'
df = pd.read_csv(path)
df

,rxn_smiles,dE0,dHrxn298
0,Cc1nnon1>>CC(N=O)=[N+]=[N-],48.61085,26.77621
1,CC(N=O)=[N+]=[N-]>>Cc1nnon1,21.83464,-26.77621
2,Cc1nnon1>>CN=C=NN=O,74.02980,28.79099
3,CN=C=NN=O>>Cc1nnon1,45.23881,-28.79099
4,COCCO>>C1COC1.O,97.42200,12.60220
...,...,...,...
23847,C=O.C[C@@H](O)C=O>>C[C@@H](O)COC=O,42.20295,-30.54744
23848,C[C@@H](O)COC=O>>C=CCOC=O.O,65.83112,14.48350
23849,C=CCOC=O.O>>C[C@@H](O)COC=O,51.34762,-14.48350
23850,C[C@@H](O)COC=O>>CC(=O)COCO,52.57819,15.06159


# Create splits

In [3]:
def create_new_indices(old_indices):
    new_indices = np.zeros(len(old_indices)*2) 
    for i, index in enumerate(old_indices):
        new_indices[2*i] = int(index*2)
        new_indices[2*i + 1] = int(index*2 +1)
    
    return new_indices.astype(int)

## Create random split the wrong way
- randomly shuffle the new data that DOES include adding the reverse reactions

In [4]:
seed = 0   # random seed for reproducability
sizes=(0.85, 0.05, 0.1)  # train, val, test sizes
total_num_datapoints = len(df)
print(f'Using {total_num_datapoints} datapoints...')

random = Random(seed)
indices = list(range(total_num_datapoints))
random.shuffle(indices)

train_size = int(sizes[0] * total_num_datapoints)
train_val_size = int((sizes[0] + sizes[1]) * total_num_datapoints)

train = indices[:train_size]
val = indices[train_size:train_val_size]
test = indices[train_val_size:]

Using 23852 datapoints...


In [5]:
indices = [[train, val, test]]
with open(f'../data/ccsdtf12/canonicalized_smiles/ccsdtf12_dz_canonical/random_split/ccsdtf12_random_split_seed_{seed}_{len(train)}.pkl', 'wb') as f:
    pkl.dump(indices, f)

In [6]:
df.iloc[indices[0][0], :].to_csv(f'ccsdtf12_random_split_seed_{seed}_train.csv', index=False)
df.iloc[indices[0][1], :].to_csv(f'ccsdtf12_random_split_seed_{seed}_val.csv', index=False)
df.iloc[indices[0][2], :].to_csv(f'ccsdtf12_random_split_seed_{seed}_test.csv', index=False)

## Create random split the correct way
- randomly shuffle the original data. do NOT use the data after adding the reverse since that is useless

In [12]:
seed = 0   # random seed for reproducability
sizes=(0.85, 0.05, 0.1)  # train, val, test sizes
total_num_datapoints = int(len(df) / 2)
print(f'Using {total_num_datapoints} datapoints...')

random = Random(seed)
indices = list(range(total_num_datapoints))
random.shuffle(indices)

train_size = int(sizes[0] * total_num_datapoints)
train_val_size = int((sizes[0] + sizes[1]) * total_num_datapoints)

train = indices[:train_size]
val = indices[train_size:train_val_size]
test = indices[train_val_size:]

Using 11926 datapoints...


In [13]:
# multiply by 2
train_indices_new = create_new_indices(train)
val_indices_new = create_new_indices(val)
test_indices_new = create_new_indices(test)

print(f'len(val): {len(val)}')
print(f'len(val_indices_new): {len(val_indices_new)}')

len(val): 596
len(val_indices_new): 1192


In [14]:
# verify that doubling takes the correct indices
df.iloc[val_indices_new, :]

,rxn_smiles,dE0,dHrxn298
1358,Cc1cc[nH]c1>>CC1=C[NH2+][C-]=C1,84.89180,71.91494
1359,CC1=C[NH2+][C-]=C1>>Cc1cc[nH]c1,12.97686,-71.91494
12454,O=C1CC=CN1>>[H]/N=C1/CC=CO1,117.25221,16.52157
12455,[H]/N=C1/CC=CO1>>O=C1CC=CN1,100.73064,-16.52157
9700,[H]/N=C/N(C)C=O>>C/N=C/NC=O,43.47672,9.72335
...,...,...,...
10287,C.CCC=C=O>>CCC(=O)CC,65.30747,-24.47287
17892,CCN1C[C@H]1CO>>C[CH-][N+]1=C[C@H]1CO.[H][H],83.24765,71.79146
17893,C[CH-][N+]1=C[C@H]1CO.[H][H]>>CCN1C[C@H]1CO,11.45619,-71.79146
15758,C/C(=N\O)[C@H]1CO1>>C=C(NO)[C@H]1CO1,77.92283,15.09075


In [15]:
indices = [[train_indices_new, val_indices_new, test_indices_new]]
with open(f'../data/ccsdtf12/canonicalized_smiles/ccsdtf12_dz_canonical/random_split_using_fwd/ccsdtf12_random_split_using_fwd_reactants_seed_{seed}_{len(train_indices_new)}.pkl', 'wb') as f:
    pkl.dump(indices, f)

In [16]:
df.iloc[indices[0][0], :].to_csv(f'ccsdtf12_random_split_using_fwd_seed_{seed}_train.csv', index=False)
df.iloc[indices[0][1], :].to_csv(f'ccsdtf12_random_split_using_fwd_seed_{seed}_val.csv', index=False)
df.iloc[indices[0][2], :].to_csv(f'ccsdtf12_random_split_using_fwd_seed_{seed}_test.csv', index=False)

## Create Scaffold Split

In [17]:
def str_to_mol(string, explicit_hydrogens=False):
    """
    Converts an InChI or SMILES string to an RDKit molecule.

    :param string: The InChI or SMILES string.
    :param explicit_hydrogens: Whether to treat hydrogens explicitly.
    :return: The RDKit molecule.
    """
    RDKIT_SMILES_PARSER_PARAMS = Chem.SmilesParserParams()
    if string.startswith('InChI'):
        mol = Chem.MolFromInchi(string, removeHs=not explicit_hydrogens)
    else:
        # Set params here so we don't remove hydrogens with atom mapping
        RDKIT_SMILES_PARSER_PARAMS.removeHs = not explicit_hydrogens
        mol = Chem.MolFromSmiles(string, RDKIT_SMILES_PARSER_PARAMS)

    if explicit_hydrogens:
        return Chem.AddHs(mol)
    else:
        return Chem.RemoveHs(mol)

In [18]:
def generate_scaffold(mol, include_chirality=False):
    """
    Compute the Bemis-Murcko scaffold for a SMILES string.

    :param mol: A smiles string or an RDKit molecule.
    :param include_chirality: Whether to include chirality.
    :return:
    """
    mol = str_to_mol(mol) if type(mol) == str else mol
    scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=include_chirality)

    return scaffold

In [19]:
def scaffold_to_smiles(mols,
                       use_indices=False):
    """
    Computes scaffold for each smiles string and returns a mapping from scaffolds to sets of smiles.

    :param mols: A list of smiles strings or RDKit molecules.
    :param use_indices: Whether to map to the smiles' index in all_smiles rather than mapping
    to the smiles string itself. This is necessary if there are duplicate smiles.
    :return: A dictionary mapping each unique scaffold to all smiles (or smiles indices) which have that scaffold.
    """
    scaffolds = defaultdict(set)
    for i, mol in tqdm(enumerate(mols), total=len(mols)):
        scaffold = generate_scaffold(mol)
        if use_indices:
            scaffolds[scaffold].add(i)
        else:
            scaffolds[scaffold].add(mol)

    return scaffolds

In [20]:
# modify this function to just return the indices
def scaffold_split(mols,
                   sizes=(0.8, 0.1, 0.1),
                   balanced=False,
                   seed=0,
                   ):
    """
    Split a dataset by scaffold so that no molecules sharing a scaffold are in the same split.

    :param data: A MoleculeDataset or ReactionDataset.
    
    :param mols: a list of rdkit molecules
    
    :param sizes: A length-3 tuple with the proportions of data in the
    train, validation, and test sets.
    :param balanced: Try to balance sizes of scaffolds in each set, rather than just putting smallest in test set.
    :param seed: Seed for shuffling when doing balanced splitting.
    :param logger: A logger.
    :return: A tuple containing the train, validation, and test splits of the data.
    """
    assert sum(sizes) == 1

    # Split
    train_size, val_size, test_size = sizes[0] * len(mols), sizes[1] * len(mols), sizes[2] * len(mols)
    train, val, test = [], [], []
    train_scaffold_count, val_scaffold_count, test_scaffold_count = 0, 0, 0

    # Map from scaffold to index in the data
    # Only use reactant molecules for reaction datasets
    # mols = list(zip(*data.mols()))[0] if isinstance(data, ReactionDataset) else data.mols()
    
    scaffold_to_indices = scaffold_to_smiles(mols, use_indices=True)

    if balanced:  # Put stuff that's bigger than half the val/test size into train, rest just order randomly
        index_sets = list(scaffold_to_indices.values())
        big_index_sets = []
        small_index_sets = []
        for index_set in index_sets:
            if len(index_set) > val_size / 2 or len(index_set) > test_size / 2:
                big_index_sets.append(index_set)
            else:
                small_index_sets.append(index_set)
        random.seed(seed)
        random.shuffle(big_index_sets)
        random.shuffle(small_index_sets)
        index_sets = big_index_sets + small_index_sets
    else:  # Sort from largest to smallest scaffold sets
        index_sets = sorted(list(scaffold_to_indices.values()),
                            key=lambda index_set: len(index_set),
                            reverse=True)

    for index_set in index_sets:
        if len(train) + len(index_set) <= train_size:
            train += index_set
            train_scaffold_count += 1
        elif len(val) + len(index_set) <= val_size:
            val += index_set
            val_scaffold_count += 1
        else:
            test += index_set
            test_scaffold_count += 1

    
    print(f'Total scaffolds = {len(scaffold_to_indices):,} | '
          f'train scaffolds = {train_scaffold_count:,} | '
          f'val scaffolds = {val_scaffold_count:,} | '
          f'test scaffolds = {test_scaffold_count:,}')
    
    # log_scaffold_stats(data, index_sets, logger=logger)
    
    return train, val, test
    
#     # Map from indices to data
#     train = [data[i] for i in train]
#     val = [data[i] for i in val]
#     test = [data[i] for i in test]

#     if isinstance(data, ReactionDataset):
#         return ReactionDataset(train), ReactionDataset(val), ReactionDataset(test)
#     else:
#         return MoleculeDataset(train), MoleculeDataset(val), MoleculeDataset(test)

In [21]:
df.rxn_smiles.values[::2].shape

(11926,)

In [22]:
mols = [str_to_mol(rsmi.split('>>')[0], explicit_hydrogens=True) for rsmi in df.rxn_smiles.values[::2]]
len(mols)

11926

In [23]:
seed = 0  # 0 3 5 6 42
train_indices, val_indices, test_indices = scaffold_split(mols,
                                                          sizes=(0.85, 0.05, 0.1),
                                                          balanced=True,
                                                          seed=seed)
len(train_indices), len(val_indices), len(test_indices)

100%|██████████| 11926/11926 [00:00<00:00, 12022.46it/s]

Total scaffolds = 205 | train scaffolds = 160 | val scaffolds = 14 | test scaffolds = 31


(10137, 593, 1196)

In [24]:
# multiply by 2
train_indices_new = create_new_indices(train_indices)
val_indices_new = create_new_indices(val_indices)
test_indices_new = create_new_indices(test_indices)

print(f'len(val): {len(val_indices)}')
print(f'len(val_indices_new): {len(val_indices_new)}')

len(val): 593
len(val_indices_new): 1186


In [25]:
# verify that doubling takes the correct indices
df.iloc[val_indices_new, :]

,rxn_smiles,dE0,dHrxn298
17476,O=Cc1ccon1>>[O+]#CC1=CCO[N-]1,128.20135,76.24354
17477,[O+]#CC1=CCO[N-]1>>O=Cc1ccon1,51.95781,-76.24354
17478,O=Cc1ccon1>>O[C+]=C1C=CO[N-]1,81.20890,51.60941
17479,O[C+]=C1C=CO[N-]1>>O=Cc1ccon1,29.59949,-51.60941
17480,O=Cc1ccon1>>[CH-]=[O+]c1ccon1,97.90292,69.78625
...,...,...,...
15931,[H]/N=C1\OC=C1NC>>[H]/N=C1\OC[C@@]12CN2,104.61987,-4.31981
5572,O1[C@H]2[C@@H]1[C@H]1O[C@@H]21>>[CH-]=[O+][C@H...,79.44946,51.24131
5573,[CH-]=[O+][C@H]1[C@H]2O[C@H]21>>O1[C@H]2[C@@H]...,28.20815,-51.24131
5574,O1[C@H]2[C@@H]1[C@H]1O[C@@H]21>>[C-]1=[O+]C[C@...,53.31398,-8.72490


In [26]:
indices = [[train_indices_new, val_indices_new, test_indices_new]]
with open(f'../data/ccsdtf12/canonicalized_smiles/ccsdtf12_dz_canonical/scaffold_split/ccsdtf12_scaffold_split_using_fwd_reactants_seed_{seed}_{len(train_indices_new)}.pkl', 'wb') as f:
    pkl.dump(indices, f)

In [29]:
df.iloc[indices[0][0], :].to_csv(f'ccsdtf12_scaffold_split_seed_{seed}_train.csv', index=False)
df.iloc[indices[0][1], :].to_csv(f'ccsdtf12_scaffold_split_seed_{seed}_val.csv', index=False)
df.iloc[indices[0][2], :].to_csv(f'ccsdtf12_scaffold_split_seed_{seed}_test.csv', index=False)